In [ ]:
library(STged)
library(DOTr)
library(ggplot2)
library(SeuratObject)
library(Seurat)
library(presto)
library(dplyr)
library(SeuratDisk)
library(Matrix)

In [ ]:
sc <- readRDS('data/spatial/processed_data/tonsil_scRNA_new.rds')
#genes <- readRDS('data/spatial/markers/gene.rds')
donor <-'BCLL-8-T'
file <- paste0('data/spatial/ST_output/', donor, '.rds')
model.est <- readRDS(file)

genes <- rownames(model.est$F_list[[1]])
gene_keep <- intersect(genes, rownames(sc))
sc <- subset(sc, features = gene_keep)
#sc <- subset(sc, subset = !annotation_level_1 %in% c("preTC", "preBC"))
sc

In [ ]:
sc <- NormalizeData(sc)
set.seed(42)
markers <- FindAllMarkers(
  object = sc,
  assay = "RNA",                          
  group.by = "cell_type",      
  only.pos = TRUE, 
  #max.cells.per.ident = 200
)
saveRDS(markers,'data/spatial/markers/markers.rds')
head(markers,20)

In [ ]:
all_mat <- list()
all_labels <- list()

for (ct in names(model.est$F_list)) {
  mat <- as.matrix(model.est$F_list[[ct]])  #gene*spot
  colnames(mat) <- paste0(colnames(mat),'_',ct)
  all_mat[[ct]] <- mat
  all_labels[[ct]] <- rep(ct, ncol(mat))
  
}
mat <- do.call(cbind, all_mat) 
mat<- Matrix(mat, sparse = TRUE)
label <- unlist(all_labels)

pseudo_seurat <- CreateSeuratObject(counts = mat)
pseudo_seurat$cell_type <- as.vector(label)
pseudo_seurat

In [ ]:
pseudo_seurat <- NormalizeData(pseudo_seurat)
markers <- FindAllMarkers(
  object = pseudo_seurat,
  assay = "RNA",                         
  group.by = "cell_type",      
  only.pos = TRUE 
)
mk_file <- paste0('data/spatial/markers/', donor, '.rds')
saveRDS(markers,file = mk_file)